In [ ]:
!pip install --upgrade datasets

In [ ]:
import json
from datasets import load_dataset



meu_dataset = load_dataset('json', data_files={
    'train': '/home/cecilia/Documentos/PIBIC/Fase1/dados_llms/train_novo.json',
    'validation': '/home/cecilia/Documentos/PIBIC/Fase1/dados_llms/val_novo.json',
    'test': '/home/cecilia/Documentos/PIBIC/Fase1/dados_llms/test_novo.json'
})


In [ ]:
dataset_test = meu_dataset['test'][0:10]
test_inputs = dataset_test['input']
test_target = dataset_test['target']

In [ ]:

def extrair_apenas_aspecto(exemplo):
    target_original = exemplo['target']
    aspectos = []

    partes = target_original.split(' [SEP] ')
    for p in partes:
        aspecto = p.split('|')[0].strip()
        if aspecto:
            aspectos.append(aspecto)


    exemplo['target_limpo'] = ' | '.join(sorted(list(set(aspectos))))
    return exemplo

meu_dataset = meu_dataset.map(extrair_apenas_aspecto)

In [ ]:
import requests
import json
from google.colab import userdata



def chamar_api(frase):
  api_key = userdata.get('API')

  url = 'https://openrouter.ai/api/v1/chat/completions'

  parte_fixa = f"""
  Your task is to extract the Aspect terms from a sentence.
    - Return ONLY the terms separated by ' | '.
    - If there is only one, return only it.
    - If it is implicit, return 'implicit'.
    - Do not explain anything.

  Example:

  Input:  The food was excellent as well as service.
  Aspect: food | service

  """

  prompt = f"""{parte_fixa}
  ###Question###
  Input: {frase}.
  Target:"""

  headers = {
        "Authorization": f"Bearer {api_key}", 
        "Content-Type": "application/json"
    }

  payload = {
      "model": "meta-llama/llama-3.1-70b-instruct",
      "messages": [{"role": "user", "content": prompt}],
      "temperature": 0,
      "max_tokens": 50,
      "top_p": 1
    }

  try:

    response = requests.post(url, headers=headers, json=payload)

    if response.status_code == 200:
      return response.json()['choices'][0]['message']['content'].strip()

    else:
      return f'Erro: {response.status_code}'

  except Exception as e:
    return f"Falha na conexão: {e}"

In [ ]:

def calcular_todas_as_metricas_corrigido(gabarito_str, ia_str):
    def limpar(texto):
      if not isinstance(texto, str):
        texto = str(texto)
      return set([a.strip().lower() for a in texto.replace('|', ',').split(',') if a.strip()])


    set_gabarito = limpar(gabarito_str)
    set_ia = limpar(ia_str)

    print(f"GABARITO: {set_gabarito} | IA: {set_ia}")

    if not set_gabarito:
        return {"Precisão": 0, "Recall": 0, "F1-Score": 0, "Acurácia": 0}

    acertos = set_gabarito.intersection(set_ia)
    n_acertos = len(acertos)

    recall = n_acertos / len(set_gabarito)
    precisao = n_acertos / len(set_ia) if len(set_ia) > 0 else 0
    f1 = (2 * precisao * recall) / (precisao + recall) if (precisao + recall) > 0 else 0


    acuracia = 1.0 if set_gabarito == set_ia else 0

    return {
        "Precisão": precisao,
        "Recall": recall,
        "F1-Score": f1,
        "Acurácia": acuracia
    }



In [ ]:

resultados_pibic_final = []
dataset_final = meu_dataset['test']

for i in range(len(dataset_final)):
    frase_original = dataset_final[i]['input']
    gabarito_oficial = dataset_final[i]['target_limpo']

    predicao = chamar_api(frase_original)

    metricas = calcular_todas_as_metricas_corrigido(gabarito_oficial, predicao)

    resultados_pibic_final.append({
        "Frase": frase_original,
        "Gabarito": gabarito_oficial,
        "IA": predicao,
        "Precisao": metricas["Precisão"],
        "Recall": metricas["Recall"],
        "F1": metricas["F1-Score"],
        "Acuracia": metricas["Acurácia"]
    })

import pandas as pd
df_final = pd.DataFrame(resultados_pibic_final)
print("\n--- MÉDIAS FINAIS DO EXPERIMENTO ---")
print(df_final[['Precisao', 'Recall', 'F1', 'Acuracia']].mean())

Valores para o dataset 'test.json' (que contém mais de um aspecto por frase):
Precisao    0.508715
Recall      0.542484
F1          0.515995
Acuracia    0.372549
